# Decision trees: práctica y ejercicios

Predigamos muerte en el Titanic con un decision tree y comparemos con Logistic Regression

In [ ]:
import pandas as pd

## Load data

In [ ]:
df = pd.read_csv("./datasets/titanic_kaggle/train.csv")

In [ ]:
df = df.drop("PassengerId", axis=1)

## Feature engineering

In [ ]:
df.isna().sum()

In [ ]:
df['Age'] = df['Age'].fillna(df.Age.mean())
df['Fare'] = df['Fare'].fillna(df['Fare'].median())

In [ ]:
df['Embarked'] = df['Embarked'].fillna("Other")
df['Cabin'] = df['Cabin'].fillna("Other")

In [ ]:
df.head()

In [ ]:
# antes que tirarlo, me quedo la longitud
df.Ticket = df.Ticket.apply(len)

In [ ]:
df.head()

In [ ]:
df.Sex = df.Sex.map({"male": 0, "female": 1})

In [ ]:
df = pd.get_dummies(df, columns=['Embarked'])

In [ ]:
df.head()

In [ ]:
df.Cabin.nunique()

In [ ]:
df.Cabin = df.Cabin.str[0]

In [ ]:
df.Cabin.nunique()

In [ ]:
df.head()

mean encoding: asignamos a cada cabina la **media de supervivencia** de esa cabina

In [ ]:
cabin_means = df.groupby('Cabin')["Survived"].mean()

In [ ]:
cabin_means

In [ ]:
df['Cabin'] = df['Cabin'].map(cabin_means)

In [ ]:
df.head()

In [ ]:
# antes que tirar el nombre, quedémonos el título
df.Name = df['Name'].str.extract(r',\s*([^\.]+)\.')

In [ ]:
name_means = df.groupby("Name").Survived.mean()

In [ ]:
name_means.sort_values()

In [ ]:
df.Name = df.Name.map(name_means)

In [ ]:
df.head()

## Train model

In [ ]:
from sklearn.tree import DecisionTreeClassifier

In [ ]:
tree = DecisionTreeClassifier(max_depth=7, min_samples_split=15)

In [ ]:
X = df.drop("Survived", axis=1)
y = df.Survived

In [ ]:
tree.fit(X, y)

## Predict and submit

### preprocess test like train

Es tedioso, pero debemos hacerlo

In [ ]:
df_test = pd.read_csv("./datasets/titanic_kaggle/test.csv")

In [ ]:
submission = df_test[["PassengerId"]].copy()
del df_test["PassengerId"]

In [ ]:
df_test['Age'] = df_test['Age'].fillna(df.Age.mean())
df_test['Fare'] = df_test['Fare'].fillna(df['Fare'].median())

In [ ]:
df_test['Embarked'] = df_test['Embarked'].fillna("Other")
df_test['Cabin'] = df_test['Cabin'].fillna("Other")

In [ ]:
df_test.Ticket = df_test.Ticket.apply(len)

In [ ]:
df_test.Sex = df_test.Sex.map({"male": 0, "female": 1})

In [ ]:
df_test = pd.get_dummies(df_test, columns=['Embarked'])

In [ ]:
df_test.Cabin = df_test.Cabin.str[0]
df_test.Cabin = df_test.Cabin.fillna("Other")

In [ ]:
df_test['Cabin'] = df_test['Cabin'].map(cabin_means)

In [ ]:
df_test.Name = df_test['Name'].str.extract(r',\s*([^\.]+)\.')

In [ ]:
df_test.Name = df_test.Name.map(name_means)

In [ ]:
df_test.head()

In [ ]:
X_test = df_test.copy()

In [ ]:
X.head()

In [ ]:
X_test.head()

falta una columna, hay que crearla

In [ ]:
X_test["Embarked_Other"] = 0

In [ ]:
X_test.head()

el orden de las columnas con qué entrenamos ha de ser el mismo que  
el orden de las columnas cuando predecimos

In [ ]:
X_test = X_test[X.columns]

In [ ]:
X.head()

In [ ]:
X_test.head()

### predict

In [ ]:
preds_test = tree.predict(X_test)

In [ ]:
preds_test[:30]

In [ ]:
submission["Survived"] = preds_test

In [ ]:
submission.head()

In [ ]:
import datetime

In [ ]:
this_moment = str(datetime.datetime.now())

In [ ]:
this_moment

In [ ]:
submission.to_csv(f"./datasets/titanic_kaggle/submission_{this_moment}.csv", index=False)